# The Arithmetic of a Label

Companion notebook for the [Cardinality and the Cost of a Label](https://codemia.io/courses/observability_and_delivery/obs_cardinality_and_the_cost_of_a_label) lesson on Codemia.

**What you will build**: the spreadsheet your monitoring vendor wishes you would build. You will multiply label cardinalities
into series counts, turn series into memory and disk, add one high cardinality label and watch the
number leave the building, then simulate the deploy churn that makes a steady fleet grow series all
day. The last section works out what actually fits in a 32 GB server and which of the fixes buys
back the most.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)

# The constants everything below is built on. These are Prometheus defaults and commonly
# measured figures, not laws of nature, but they are the right order of magnitude.
HEAD_BYTES_PER_SERIES = 3_500      # in-memory cost of one active series
DISK_BYTES_PER_SAMPLE = 1.7        # after delta-of-delta and XOR compression
SCRAPE_INTERVAL_S = 15
SAMPLES_PER_DAY = 86_400 // SCRAPE_INTERVAL_S

## Concept recap

A time series is identified by its metric name and the exact set of label values attached to it.
`http_requests_total{method="GET", status="200", endpoint="/checkout", instance="web-07"}` is one
series. Change any single value and it is a different series with its own index entry, its own
memory and its own chunks on disk.

So the series count of a metric is the product of its label cardinalities, not the sum. This is the
only piece of arithmetic in observability that regularly costs companies six figures, because
multiplication does not feel like what is happening when you add one field to a label set.

## One metric, priced

In [ ]:
def series_count(labels, bucket_series=1):
    """Series for one metric: the product of label cardinalities, times the series per sample."""
    product = 1
    for cardinality in labels.values():
        product *= cardinality
    return product * bucket_series


def price(series, name=''):
    head_gb = series * HEAD_BYTES_PER_SERIES / 2 ** 30
    disk_gb_day = series * SAMPLES_PER_DAY * DISK_BYTES_PER_SAMPLE / 2 ** 30
    if name:
        print(f'{name:>38}{series:>12,}{head_gb:>11.2f} GB{disk_gb_day:>12.2f} GB/day')
    return head_gb, disk_gb_day


BASE = {'method': 6, 'status': 8, 'endpoint': 40, 'instance': 120}
HISTOGRAM_SERIES = 14              # 12 bucket boundaries plus _sum and _count

print(f'{"metric":>38}{"series":>12}{"head memory":>14}{"disk":>16}')
price(series_count({'instance': 120}), 'process_cpu_seconds_total')
price(series_count(BASE), 'http_requests_total')
price(series_count(BASE, HISTOGRAM_SERIES), 'http_request_duration_seconds (hist)')
print()
print('the label set, one factor at a time:')
running = 1
for label, cardinality in BASE.items():
    running *= cardinality
    print(f'  x {label:<10} ({cardinality:>3}) -> {running:>10,} series')

The histogram is the line to stare at. A latency histogram is not one series per label combination,
it is one per bucket boundary plus a sum and a count, so it multiplies the whole product by roughly
fourteen. Most of the series in a mature Prometheus are histogram buckets, and the cheapest
cardinality win available to most teams is deleting bucket boundaries nobody reads.

## The product is an upper bound, and the gap is large

In [ ]:
ENDPOINTS = 40
INSTANCES = 120

# Which combinations actually occur: an endpoint serves one or two methods, and each of those
# returns a handful of distinct status codes rather than all eight.
methods_per_endpoint = rng.choice([1, 2, 3], size=ENDPOINTS, p=[0.55, 0.35, 0.10])
realized_per_instance = 0
for e in range(ENDPOINTS):
    for _ in range(methods_per_endpoint[e]):
        realized_per_instance += int(rng.choice([2, 3, 4, 5], p=[0.35, 0.35, 0.20, 0.10]))

product = series_count(BASE, HISTOGRAM_SERIES)
realized = realized_per_instance * INSTANCES * HISTOGRAM_SERIES

print(f'label cardinality product       {product:>12,} series')
print(f'combinations that actually occur {realized:>11,} series')
print(f'the product overstates reality by {product / realized:>8.1f}x')
print()
price(realized, 'what this service really costs')
print()
print(f'per instance: {realized_per_instance:,} label combinations across '
      f'{ENDPOINTS} endpoints')

Two lessons sit in that ratio, and they pull in opposite directions.

The first is that you should measure rather than multiply. A GET on `/checkout` never returns 405,
and a sparse label space is the normal case, so a capacity estimate built from the product is wrong
by an order of magnitude. `count({__name__="http_request_duration_seconds_bucket"})` answers the
question in a second and no spreadsheet does.

The second is that the product is still the number to fear, because it is what you get the day a
genuinely independent label arrives. Sparsity protects you only while the label set stays
correlated. Everything from here on uses the realized figure as the starting point.

## Which labels actually multiply

In [ ]:
# (name, cardinality, independent of the labels already present?)
CANDIDATES = [
    ('availability_zone', 12, False),     # an instance sits in exactly one zone
    ('region', 4, False),                 # likewise
    ('pod_name', 480, False),             # in a Kubernetes fleet this replaces instance
    ('cache_hit', 2, True),
    ('customer_tier', 3, True),
    ('customer_id', 2_400, True),
    ('user_id', 50_000, True),
]

print(f'{"new label":>20}{"cardinality":>13}{"independent":>13}{"factor":>9}'
      f'{"series":>16}{"head memory":>14}')
results = []
for name, cardinality, independent in CANDIDATES:
    factor = cardinality if independent else 1
    s = realized * factor
    head = s * HEAD_BYTES_PER_SERIES / 2 ** 30
    head_str = f'{head:,.1f} GB' if head < 1024 else f'{head / 1024:,.1f} TB'
    results.append((name, s, independent))
    print(f'{name:>20}{cardinality:>13,}{str(independent):>13}{factor:>9,}'
          f'{s:>16,}{head_str:>14}')

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.bar([n for n, _, _ in results], [s for _, s, _ in results],
        color=['tab:blue' if not ind else 'tab:green' if s < 1e7 else 'tab:red'
               for _, s, ind in results])
plt.axhline(realized, color='black', linestyle='--',
            label=f'before the new label ({realized:,} series)')
plt.yscale('log')
plt.ylabel('series for this one metric')
plt.xticks(rotation=25, ha='right')
plt.title('Blue labels are free. Green are affordable. Red is an outage.')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.show()

The blue bars are the surprise, and they are worth more than the red ones because they are the bars
people get wrong in the cautious direction. A zone label on a metric that already carries the
instance costs nothing: an instance is in exactly one zone, so the label is functionally determined
by a label you already have and no new combination appears. Teams refuse these labels on cardinality
grounds all the time and then spend an incident hand mapping instance names to zones.

The test is not how many values a label has. It is how many values it has *for a fixed value of the
labels already present*. A `pod_name` is free if it replaces `instance` and ruinous if it is added
alongside it while pods are being recreated, which is the churn case the next section measures.

And the red bars are red for a reason that no amount of cleverness fixes: a user id is independent
of everything else on the metric, so it multiplies, and it multiplies by a number that grows with
your success.

## Churn: a steady fleet that grows series all day

In [ ]:
RETENTION_HOURS = 2            # how long a series lingers in the head block after its last sample


def live_generations(deploys_per_day):
    """How many generations of pod-labelled series are resident at once."""
    if deploys_per_day == 0:
        return 1
    spacing_hours = 24 / deploys_per_day
    return 1 + int(RETENTION_HOURS // spacing_hours)


def active_series_curve(deploys_per_day, base):
    hours = np.arange(0, 24, 0.1)
    deploy_times = np.linspace(0, 24, deploys_per_day, endpoint=False) if deploys_per_day else []
    out = []
    for h in hours:
        live = sum(1 for d in deploy_times if d <= h and h - d <= RETENTION_HOURS)
        out.append(max(live, 1) * base)
    return hours, np.array(out)


plt.figure(figsize=(10, 4.5))
for deploys in [1, 6, 12, 48]:
    hours, s = active_series_curve(deploys, realized)
    plt.plot(hours, s / 1e6, label=f'{deploys} deploys/day, peak {s.max() / 1e6:.2f} M series')
plt.xlabel('hour of the day')
plt.ylabel('active series, millions')
plt.title('Series churn: the fleet is the same size all day')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f'{"deploys/day":>12}{"spacing":>12}{"generations":>13}{"peak series":>14}{"head memory":>14}')
for deploys in [1, 6, 12, 24, 48]:
    g = live_generations(deploys)
    peak = g * realized
    print(f'{deploys:>12}{24 / deploys:>10.1f} h{g:>13}{peak:>14,}'
          f'{peak * HEAD_BYTES_PER_SERIES / 2 ** 30:>11.2f} GB')

Nothing scaled up. The same pods serve the same traffic, and the memory doubles, trebles or
quintuples because series that stopped receiving samples are still resident until they age out of
the head block.

The rule the table makes visible: generations overlap when deploys are closer together than the head
block retention. One deploy a day costs nothing at all; a deploy every half hour against a two hour
head block carries five generations at once. This is the mechanism behind the complaint that
Prometheus memory grows on deploy days, and the reason continuous deployment and a `pod_name` label
are an expensive pairing.

It also explains a confusing operational detail. Peak memory does not follow peak traffic. It follows
rollouts, which is a useful thing to know at three in the morning when memory is climbing and
request rate is flat.

## What fits in a server

In [ ]:
SERVER_GB = 32
USABLE_FRACTION = 0.6              # leave room for queries, compaction and the operating system

budget = int(SERVER_GB * USABLE_FRACTION * 2 ** 30 / HEAD_BYTES_PER_SERIES)
print(f'{SERVER_GB} GB server, {USABLE_FRACTION:.0%} for the head block: '
      f'{budget:,} active series')
print()

SERVICE = {
    'request histogram': realized,
    'request counters': realized // HISTOGRAM_SERIES,
    'process metrics (40 x instance)': 40 * INSTANCES,
    'go runtime metrics (60 x instance)': 60 * INSTANCES,
    'queue depth per topic (90 x instance)': 90 * INSTANCES,
}
per_service = sum(SERVICE.values())
for name, s in SERVICE.items():
    print(f'{name:>40}{s:>12,}{s / per_service:>9.1%}')
print(f'{"one service, one generation":>40}{per_service:>12,}')
print()

print(f'{"deploys/day":>12}{"series/service":>16}{"services that fit":>20}')
for deploys in [0, 6, 12, 48]:
    live = per_service * live_generations(deploys)
    print(f'{deploys:>12}{live:>16,}{budget // live:>20}')
print()
print('and with a customer_id label on the histogram:')
with_customer = per_service + realized * (2_400 - 1)
print(f'  {with_customer:,} series per service, '
      f'{with_customer / budget:.0f}x the whole server')

A single Prometheus with a sensible label set holds a double digit number of services, and that
number is divided by how often you deploy. The same server holds a fraction of one service the
moment a customer identifier appears on a histogram.

That is the entire capacity conversation, and it is why the useful question in a design review is
never whether there is enough monitoring capacity. It is which labels the capacity is being spent
on.

## The three fixes, priced

In [ ]:
print(f'a dashboard panel that sums across instances touches {realized:,} series')

# Fix 1: aggregate at write time with a recording rule that drops the instance label.
rule_series = realized // INSTANCES
print(f'the same panel against a recording rule touches {rule_series:,} series '
      f'({realized / rule_series:.0f}x less work per refresh)')
print(f'and the rule itself costs {rule_series:,} new series, '
      f'{rule_series * HEAD_BYTES_PER_SERIES / 2 ** 20:.0f} MB')
print()

# Fix 2: drop bucket boundaries nobody reads.
for buckets in [14, 10, 8, 6]:
    s = realized // HISTOGRAM_SERIES * buckets
    print(f'{buckets:>3} series per histogram -> {s:>10,} series, '
          f'{s * HEAD_BYTES_PER_SERIES / 2 ** 30:>5.2f} GB')
print()

# Fix 3: move the high cardinality dimension off the metric entirely.
EXEMPLAR_RATE = 0.001
requests_per_day = 40_000_000
as_label = realized * 50_000
print(f'user_id as a label: {as_label:,} series, '
      f'{as_label * HEAD_BYTES_PER_SERIES / 2 ** 40:.1f} TB of memory')
print(f'user_id on exemplars at {EXEMPLAR_RATE:.1%} of {requests_per_day:,} requests: '
      f'{int(requests_per_day * EXEMPLAR_RATE):,} records a day, and zero new series')

The three fixes are a sequence rather than a menu, in increasing order of how much thinking they
need. Trimming buckets is a configuration change nobody will notice. A recording rule adds a small
amount of new cost and makes a dashboard panel an order of magnitude cheaper to refresh, which
matters because panels refresh unattended on wall displays for years. Moving a dimension onto
exemplars or traces is a design change, and it is the only one of the three that makes a genuinely
high cardinality question answerable at all.

What none of them do is let you keep the label. That is worth saying out loud in the review, because
the counter proposal is always a bigger server, and the arithmetic above is what tells you how much
bigger.

## What to take away

- Series are the product of label cardinalities, and a histogram multiplies that product again. The
  arithmetic is trivial and almost never done before the label is merged.
- The product is an upper bound. Real label spaces are sparse, so measure the series count instead of
  estimating it, and keep the product in mind as the number a new independent label unlocks.
- A label that is functionally determined by a label you already have is free. Zone, region and
  cluster usually fall in that class, and refusing them on cardinality grounds is a mistake that
  costs you during incidents.
- Memory is per active series, so churn is real money: generations overlap whenever deploys are
  closer together than the head block retention, and peak memory follows rollouts rather than
  traffic.
- Trim buckets, aggregate with recording rules, move high cardinality dimensions onto exemplars and
  traces. None of those recovers a user id label, because nothing does.